# Flight Route & Traffic Analysis using Live ADS-B Data - ASIA

## 1. Import Libraries

In [1]:
import requests
import pandas as pd
from pathlib import Path

## 2. Pull Live Aircraft Data from OpenSky API

In [2]:
url = "https://opensky-network.org/api/states/all"

#Assign paramaters through Asian region latitude & longtitude 
params = {
    "lamin": -10,
    "lomin": 25,
    "lamax": 55,
    "lomax": 150,
}

response = requests.get(url, params=params, timeout=30)

print(response.status_code)

200


## 3. API Response Check

In [3]:
data = response.json()

print(data.keys()) 

dict_keys(['time', 'states'])


In [4]:
print(data["time"]) #API Timestamp
print(type(data["states"])) #Confirm if Aircraft data in list
print(len(data["states"])) #Number of Aircraft rows in Asia 

1783039232
<class 'list'>
1351


In [5]:
data["states"][0] #Check first aircraft row record 

['801648',
 'IGO252V ',
 'India',
 1783039123,
 1783039123,
 77.0898,
 28.5604,
 None,
 True,
 1.29,
 8.44,
 None,
 None,
 None,
 None,
 False,
 0]

## 4. Create DataFrame

In [6]:
#Assign aircraft status data into identifying columns for DataFrame
columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source",
]

df = pd.DataFrame(data["states"], columns=columns)

df.head()

,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,801648,IGO252V,India,1783039123,1783039123,77.0898,28.5604,NaN,True,1.29,8.44,NaN,None,NaN,NaN,False,0
1,50049b,T7BLV,San Marino,1783039231,1783039231,100.7549,0.9563,11582.40,False,215.41,334.84,0.00,None,12329.16,NaN,False,0
2,801622,IGO2121,India,1783039223,1783039232,73.0635,18.8691,2164.08,False,144.29,62.16,13.66,None,2156.46,NaN,False,0
3,801689,AXB2018,India,1783039230,1783039230,77.6952,13.5744,3992.88,False,161.12,15.37,9.75,None,4107.18,NaN,False,0
4,801681,AXB2064,India,1783039229,1783039229,78.0438,13.5291,5212.08,False,191.67,64.40,7.48,None,5295.90,NaN,False,0


## 5. Data Quality Check

In [10]:
#Assign & run function to inspect data quality 
def check_dataframe(df):
    print("Info:")
    df.info()
    
    print("\nMissing values:")
    print(df.isna().sum())
    
    print("\nDuplicate rows:")
    print(df.duplicated().sum())
    
    print("\nFirst 5 rows:")
    display(df.head())

check_dataframe(df)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1351 entries, 0 to 1350
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   icao24           1351 non-null   str    
 1   callsign         1351 non-null   str    
 2   origin_country   1351 non-null   str    
 3   time_position    1351 non-null   int64  
 4   last_contact     1351 non-null   int64  
 5   longitude        1351 non-null   float64
 6   latitude         1351 non-null   float64
 7   baro_altitude    1234 non-null   float64
 8   on_ground        1351 non-null   bool   
 9   velocity         1351 non-null   float64
 10  true_track       1351 non-null   float64
 11  vertical_rate    1228 non-null   float64
 12  sensors          0 non-null      object 
 13  geo_altitude     1200 non-null   float64
 14  squawk           750 non-null    str    
 15  spi              1351 non-null   bool   
 16  position_source  1351 non-null   int64  
dtypes: bool(2), float64

,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,801648,IGO252V,India,1783039123,1783039123,77.0898,28.5604,NaN,True,1.29,8.44,NaN,None,NaN,NaN,False,0
1,50049b,T7BLV,San Marino,1783039231,1783039231,100.7549,0.9563,11582.40,False,215.41,334.84,0.00,None,12329.16,NaN,False,0
2,801622,IGO2121,India,1783039223,1783039232,73.0635,18.8691,2164.08,False,144.29,62.16,13.66,None,2156.46,NaN,False,0
3,801689,AXB2018,India,1783039230,1783039230,77.6952,13.5744,3992.88,False,161.12,15.37,9.75,None,4107.18,NaN,False,0
4,801681,AXB2064,India,1783039229,1783039229,78.0438,13.5291,5212.08,False,191.67,64.40,7.48,None,5295.90,NaN,False,0


### Initial Data Quality Check Notes: 
- 1351 aircraft status entries accross 17 categories noted in dataset.
- 7 Redundant & missing data columns identified for removal: last_contact, vertical_rate, sensors, geo_altitude, squawk, spi, position_source.
- Baro_altiude has 117 missing values
- 0 duplicate rows were found in the dataset.  


In [11]:
print("\nSummary statistics:")
display(df.describe())


Summary statistics:


,time_position,last_contact,longitude,latitude,baro_altitude,velocity,true_track,vertical_rate,geo_altitude,position_source
count,1.351000e+03,1.351000e+03,1351.000000,1351.000000,1234.000000,1351.000000,1351.000000,1228.000000,1200.000000,1351.0
mean,1.783039e+09,1.783039e+09,90.464683,28.225404,7805.930470,189.086299,188.031850,0.452231,8331.523850,0.0
std,2.745354e+02,6.679058e+01,38.633945,11.585093,3809.864841,86.798102,98.084615,5.179007,4011.329912,0.0
min,1.783035e+09,1.783039e+09,25.042100,-7.769400,7.620000,0.000000,0.000000,-26.660000,0.000000,0.0
25%,1.783039e+09,1.783039e+09,51.536800,22.311900,4537.710000,146.420000,107.000000,-0.330000,4962.525000,0.0
50%,1.783039e+09,1.783039e+09,101.290200,30.423000,9448.800000,221.190000,182.470000,0.000000,10039.350000,0.0
75%,1.783039e+09,1.783039e+09,126.094600,36.761450,10972.800000,243.515000,273.900000,0.330000,11614.785000,0.0
max,1.783039e+09,1.783039e+09,144.167800,54.724600,13716.000000,1330.870000,359.560000,21.130000,14607.540000,0.0



- Latitude ranges from about -7.77 to 54.72, and longitude ranges from about 25.04 to 144.17, which fits the Asia region filter.
- There may be an outlier in velocity due to noted maximum velocity being 1330.87 m/s, which is extremely high for normal aircraft traffic.

In [18]:
#Assign columns to keep through dropping the 7 unrequired columns.
keep_columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track"
]

df_selected = df[keep_columns].copy()

check_dataframe(df_selected)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1351 entries, 0 to 1350
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   icao24          1351 non-null   str    
 1   callsign        1351 non-null   str    
 2   origin_country  1351 non-null   str    
 3   time_position   1351 non-null   int64  
 4   longitude       1351 non-null   float64
 5   latitude        1351 non-null   float64
 6   baro_altitude   1234 non-null   float64
 7   on_ground       1351 non-null   bool   
 8   velocity        1351 non-null   float64
 9   true_track      1351 non-null   float64
dtypes: bool(1), float64(5), int64(1), str(3)
memory usage: 96.4 KB

Missing values:
icao24              0
callsign            0
origin_country      0
time_position       0
longitude           0
latitude            0
baro_altitude     117
on_ground           0
velocity            0
true_track          0
dtype: int64

Duplicate rows:
0

First 5 rows:


,icao24,callsign,origin_country,time_position,longitude,latitude,baro_altitude,on_ground,velocity,true_track
0,801648,IGO252V,India,1783039123,77.0898,28.5604,NaN,True,1.29,8.44
1,50049b,T7BLV,San Marino,1783039231,100.7549,0.9563,11582.40,False,215.41,334.84
2,801622,IGO2121,India,1783039223,73.0635,18.8691,2164.08,False,144.29,62.16
3,801689,AXB2018,India,1783039230,77.6952,13.5744,3992.88,False,161.12,15.37
4,801681,AXB2064,India,1783039229,78.0438,13.5291,5212.08,False,191.67,64.40


In [15]:
#Check for Outliers in 'velocity'
df_selected.sort_values("velocity", ascending=False).head(10)

,icao24,callsign,origin_country,time_position,longitude,latitude,baro_altitude,on_ground,velocity,true_track
960,4baa44,THY3AB,Turkey,1783037029,31.9096,54.7088,9753.60,False,1330.87,295.65
518,06a2f2,QTR976,Qatar,1783039225,53.8151,24.6633,7673.34,False,655.51,216.67
1139,01025a,NMA642,Egypt,1783038218,26.3627,53.1394,11277.60,False,325.31,313.59
1047,abb6b5,FDX5163,United States,1783039226,136.6361,37.0680,10058.40,False,295.28,64.95
1068,4d0105,CLX4164,Luxembourg,1783039231,43.0939,40.9490,9448.80,False,295.22,103.50
1179,8466c7,NCA049,Japan,1783039232,137.9183,36.7665,10957.56,False,294.32,80.34
609,71c359,AAR222,Republic of Korea,1783039232,132.5982,36.3329,10668.00,False,294.08,92.71
438,406f77,BAW5,United Kingdom,1783039221,130.6439,36.7735,12489.18,False,293.76,122.05
437,406f79,BAW199,United Kingdom,1783039162,38.6178,39.9194,10660.38,False,293.65,82.55
1283,71be26,AAR102,Republic of Korea,1783039227,135.0369,36.3319,11887.20,False,293.49,78.47


In [17]:
#Check for blank/space values in text columns 
(df_selected[["callsign", "origin_country"]].apply(lambda col: col.str.strip()) == "").sum()

callsign          42
origin_country     0
dtype: int64

- Noted 2 outliers 1330.87 m/s & 655.51 m/s in comparison to remaining aircraft speeds.
- Noted 42 blank callsigns after removing spaces. 

## 6. Data Cleaning

In [20]:
#Drop missing values in 'baro_altitude'
df_cleaned = df_selected.dropna(subset=["baro_altitude"]).copy()

# Remove unusually high velocity values above 350 m/s
df_cleaned = df_cleaned[df_cleaned["velocity"] <= 350].copy()

#Remove blank callsigns
df_cleaned["callsign"] = df_cleaned["callsign"].str.strip()

df_cleaned = df_cleaned[df_cleaned["callsign"] != ""].copy()

#reset index after row removals
df_cleaned = df_cleaned.reset_index(drop=True)

check_dataframe(df_cleaned)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1208 entries, 0 to 1207
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   icao24          1208 non-null   str    
 1   callsign        1208 non-null   str    
 2   origin_country  1208 non-null   str    
 3   time_position   1208 non-null   int64  
 4   longitude       1208 non-null   float64
 5   latitude        1208 non-null   float64
 6   baro_altitude   1208 non-null   float64
 7   on_ground       1208 non-null   bool   
 8   velocity        1208 non-null   float64
 9   true_track      1208 non-null   float64
dtypes: bool(1), float64(5), int64(1), str(3)
memory usage: 86.2 KB

Missing values:
icao24            0
callsign          0
origin_country    0
time_position     0
longitude         0
latitude          0
baro_altitude     0
on_ground         0
velocity          0
true_track        0
dtype: int64

Duplicate rows:
0

First 5 rows:


,icao24,callsign,origin_country,time_position,longitude,latitude,baro_altitude,on_ground,velocity,true_track
0,50049b,T7BLV,San Marino,1783039231,100.7549,0.9563,11582.40,False,215.41,334.84
1,801622,IGO2121,India,1783039223,73.0635,18.8691,2164.08,False,144.29,62.16
2,801689,AXB2018,India,1783039230,77.6952,13.5744,3992.88,False,161.12,15.37
3,801681,AXB2064,India,1783039229,78.0438,13.5291,5212.08,False,191.67,64.40
4,4bc8d0,PGT91Q,Turkey,1783039231,31.2706,32.1018,11582.40,False,236.47,345.00


In [22]:
total_aircraft = len(df_cleaned)

print("Total cleaned aircraft records:", total_aircraft)

Total cleaned aircraft records: 1208


# 7. Save to CSV

In [21]:
# Save raw and cleaned datasets
raw_path = "../data/raw/opensky_asia_raw.csv"
cleaned_path = "../data/clean/opensky_asia_cleaned.csv"

df.to_csv(raw_path, index=False)
df_cleaned.to_csv(cleaned_path, index=False)

print("Raw data saved to:", raw_path)
print("Cleaned data saved to:", cleaned_path)

Raw data saved to: ../data/raw/opensky_asia_raw.csv
Cleaned data saved to: ../data/clean/opensky_asia_cleaned.csv


# 8. Air Traffic Analysis

In [24]:
#Summary of barometric aircraft altitude 

average_altitude = df_cleaned["baro_altitude"].mean()
min_altitude = df_cleaned["baro_altitude"].min()
max_altitude = df_cleaned["baro_altitude"].max()

print("Average altitude:", round(average_altitude, 2), "meters")
print("Minimum altitude:", round(min_altitude, 2), "meters")
print("Maximum altitude:", round(max_altitude, 2), "meters")

Average altitude: 7775.62 meters
Minimum altitude: 7.62 meters
Maximum altitude: 13716.0 meters


- The maximum altitude is  13,716 meters(approx 45,000ft) within realistic altitude range.
- Average altitude is lower as the live dataset includes aircraft at different flight phases.

In [25]:
#Summary of aircraft Velocity 
average_speed = df_cleaned["velocity"].mean()
min_speed = df_cleaned["velocity"].min()
max_speed = df_cleaned["velocity"].max()

print("Average speed:", round(average_speed, 2), "m/s")
print("Minimum speed:", round(min_speed, 2), "m/s")
print("Maximum speed:", round(max_speed, 2), "m/s")

Average speed: 204.83 m/s
Minimum speed: 0.0 m/s
Maximum speed: 325.31 m/s


- Aircraft velocity within average/ reasonable limits

In [27]:
#check count of aircraft by origin countries
origin_country_counts = df_cleaned["origin_country"].value_counts()

origin_country_counts.head(10)

origin_country
Japan                   156
China                   112
Turkey                   98
United Arab Emirates     91
India                    90
Republic of Korea        87
Qatar                    54
Thailand                 40
Malaysia                 40
Taiwan                   40
Name: count, dtype: int64

In [28]:
#check aircraft count by hour 
#convert time_position (Unix timestamp) to dataframe 
df_cleaned["time_position_datetime"] = pd.to_datetime(
    df_cleaned["time_position"],
    unit="s"
)

df_cleaned[["time_position", "time_position_datetime"]].head()

,time_position,time_position_datetime
0,1783039231,2026-07-03 00:40:31
1,1783039223,2026-07-03 00:40:23
2,1783039230,2026-07-03 00:40:30
3,1783039229,2026-07-03 00:40:29
4,1783039231,2026-07-03 00:40:31


In [31]:
df_cleaned["hour"] = df_cleaned["time_position_datetime"].dt.hour

df_cleaned[["time_position_datetime", "hour"]].head()

,time_position_datetime,hour
0,2026-07-03 00:40:31,0
1,2026-07-03 00:40:23,0
2,2026-07-03 00:40:30,0
3,2026-07-03 00:40:29,0
4,2026-07-03 00:40:31,0


-  Hourly analysis reflects the API collection time 

In [32]:
#check airborne vs on-ground aircraft 
ground_status_counts = df_cleaned["on_ground"].value_counts()

ground_status_counts

on_ground
False    1202
True        6
Name: count, dtype: int64

### API Limitation

The OpenSky live states API does not provide departure airport, arrival airport, or full route information in this endpoint. Because of this, this project does not calculate busiest departure airports or exact route paths. The analysis focuses on live aircraft position, altitude, speed, origin country, and map visualization.

## 9. Asia Flight Map

In [33]:
import folium 

In [123]:
#Generate map centered at Asia bounding. Center lat: (-10 + 55) / 2 = 22.5 , center longt:(25 + 150) / 2 = 87.5
asia_map = folium.Map(
    location=[22.5, 87.5],
    zoom_start=3,
    tiles="CartoDB dark_matter"
)

asia_map

In [124]:
#Assign function to color aircraft markers by altitude 

def altitude_color(altitude):
    if altitude < 3000:
        return "#38bdf8" #skyblue
    elif altitude < 10000:
        return "lightgreen"   # green
    else:
        return"#ff4fd8"   # pink

In [125]:

# Add aircraft markers to the map
for _, row in df_cleaned.iterrows():
    marker_color = altitude_color(row["baro_altitude"])
    
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        stroke=False,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.6,
        popup=(
            f"Callsign: {row['callsign']}<br>"
            f"Country: {row['origin_country']}<br>"
            f"Altitude: {row['baro_altitude']} m<br>"
            f"Speed: {row['velocity']} m/s"
        )
    ).add_to(asia_map)

asia_map

## Save Map: 

In [126]:
asia_map.save("../maps/opensky_asia_flight_map.html")

print("Asia flight map saved to: ../maps/opensky_asia_flight_map.html")

Asia flight map saved to: ../maps/opensky_asia_flight_map.html
